# Análise de Registros de Tempo — Pipeline ETL

Este notebook organiza a solução no formato **ETL**:

- **E — Extração:** leitura do arquivo `data.json`.
- **T — Transformação:** limpeza, agrupamentos, ordenações, rankings e cálculos percentuais.
- **L — Carga:** geração do arquivo final `result.json`.

A lógica segue as regras da atividade: ignorar registros inválidos, calcular totais por tarefa, identificar rankings e salvar a saída no formato esperado.

---

## 1. Importação das bibliotecas

Bibliotecas utilizadas para leitura, tratamento dos dados e geração do arquivo JSON final.

In [1]:
import json
import pandas as pd

---

# E — Extração

## 2. Leitura do arquivo de entrada

Nesta etapa, a aplicação lê o arquivo local `data.json`, que contém os registros de timesheet.

In [2]:
df = pd.read_json("data.json")

total_original = len(df)

print(f"Registros originais: {total_original}")
df.head()

Registros originais: 300


,userId,userName,taskId,taskName,status,minutes,date
0,6,Fernanda,110,Criar endpoint relatório,done,25,2026-03-01
1,8,Helena,106,Criar testes unitários,doing,80,2026-02-21
2,4,Diego,105,Refatorar autenticação,done,9,2026-01-04
3,3,Carla,104,Documentar API,todo,-1,2026-04-05
4,7,Gabriel,110,Criar endpoint relatório,doing,117,2026-04-28


---

# T — Transformação

## 3. Tratamento dos dados

Conforme a regra da atividade, todos os registros com `minutes <= 0` devem ser ignorados.
Também é calculada a quantidade de registros descartados.

In [3]:
df_valido = df[df["minutes"] > 0].copy()

total_valido = len(df_valido)
registros_ignorados = total_original - total_valido

print(f"Registros válidos: {total_valido}")
print(f"Registros ignorados: {registros_ignorados}")

Registros válidos: 259
Registros ignorados: 41


## 4. Cálculo do total geral de minutos

Soma de todos os minutos válidos, usada como base para os percentuais por tarefa.

In [4]:
total_minutes = df_valido["minutes"].sum()

print(f"Total de minutos gastos em todas as tarefas: {total_minutes}")

Total de minutos gastos em todas as tarefas: 28408


## 5. Total de minutos por tarefa

Agrupamento por `taskId` e `taskName`, somando os minutos válidos de cada tarefa.

> O enunciado pede agrupamento por `taskId`, mas o `taskName` é mantido para preservar a estrutura esperada do arquivo final.

In [5]:
tasks_agrupadas = (
    df_valido
    .groupby(["taskId", "taskName"], as_index=False)["minutes"]
    .sum()
    .rename(columns={"minutes": "totalMinutes"})
)

tasks_agrupadas.head()

,taskId,taskName,totalMinutes
0,101,Corrigir bug login,2088
1,102,Criar tela dashboard,2713
2,103,Ajustar layout,4047
3,104,Documentar API,2728
4,105,Refatorar autenticação,2804


## 6. Ordenação das tarefas e cálculo de percentual

As tarefas são ordenadas de forma determinística:

1. `totalMinutes` em ordem decrescente;
2. `taskId` em ordem crescente em caso de empate.

Depois, é calculado o percentual de cada tarefa sobre o total geral.

In [6]:
tasks_ordenadas = tasks_agrupadas.sort_values(
    by=["totalMinutes", "taskId"],
    ascending=[False, True]
).reset_index(drop=True)

tasks_ordenadas["percentage"] = tasks_ordenadas["totalMinutes"].apply(
    lambda x: f"{(x / total_minutes) * 100:.2f}%"
)

tasks_ordenadas["taskId"] = tasks_ordenadas["taskId"].astype(int)
tasks_ordenadas["totalMinutes"] = tasks_ordenadas["totalMinutes"].astype(int)

tasks_ordenadas

,taskId,taskName,totalMinutes,percentage
0,103,Ajustar layout,4047,14.25%
1,110,Criar endpoint relatório,3500,12.32%
2,106,Criar testes unitários,3050,10.74%
3,108,Melhorar queries,3041,10.70%
4,105,Refatorar autenticação,2804,9.87%
5,104,Documentar API,2728,9.60%
6,102,Criar tela dashboard,2713,9.55%
7,107,Implementar cache,2474,8.71%
8,101,Corrigir bug login,2088,7.35%
9,109,Ajustar responsividade,1963,6.91%


## 7. Tarefa mais trabalhada

Como as tarefas já estão ordenadas, a primeira linha representa a tarefa com maior total de minutos.

In [7]:
task_mais_trabalhada = tasks_ordenadas.iloc[0].to_dict()

tarefa_mais_trabalhada_limpa = {
    "taskId": int(task_mais_trabalhada["taskId"]),
    "taskName": task_mais_trabalhada["taskName"],
    "totalMinutes": int(task_mais_trabalhada["totalMinutes"]),
    "percentage": task_mais_trabalhada["percentage"]
}

tarefa_mais_trabalhada_limpa

{'taskId': 103,
 'taskName': 'Ajustar layout',
 'totalMinutes': 4047,
 'percentage': '14.25%'}

## 8. Top 3 tarefas por percentual

Seleciona as três tarefas com maior total de minutos, exibindo o percentual com duas casas decimais.

In [17]:
top_3_tarefas = (
    tasks_ordenadas[["taskId", "taskName", "percentage"]]
    .head(3)
    .to_dict(orient="records")
)

df_top_3_tarefas = pd.DataFrame(top_3_tarefas)
df_top_3_tarefas.head(3)

,taskId,taskName,percentage
0,103,Ajustar layout,14.25%
1,110,Criar endpoint relatório,12.32%
2,106,Criar testes unitários,10.74%


## 9. Agrupamento dos funcionários

Agrupa os registros por `userId` e `userName`, calculando:

- total de minutos trabalhados;
- lista de tarefas distintas;
- quantidade de tarefas distintas.

In [9]:
funcs_agrupadas = (
    df_valido
    .groupby(["userId", "userName"])
    .agg(
        totalMinutes=("minutes", "sum"),
        taskIds=("taskId", lambda x: sorted([int(i) for i in x.unique()]))
    )
    .reset_index()
)

funcs_agrupadas["distinctTasks"] = funcs_agrupadas["taskIds"].apply(len)
funcs_agrupadas["userId"] = funcs_agrupadas["userId"].astype(int)
funcs_agrupadas["totalMinutes"] = funcs_agrupadas["totalMinutes"].astype(int)

funcs_agrupadas.head()

,userId,userName,totalMinutes,taskIds,distinctTasks
0,1,Ana,4077,"[101, 102, 103, 104, 105, 106, 107, 108, 109, ...",10
1,2,Bruno,3408,"[101, 102, 103, 104, 105, 106, 107, 108, 109, ...",10
2,3,Carla,3842,"[101, 102, 103, 104, 105, 106, 107, 108, 109, ...",10
3,4,Diego,3350,"[101, 102, 103, 104, 105, 106, 107, 108, 109, ...",10
4,5,Eduardo,4303,"[101, 102, 103, 104, 105, 106, 107, 108, 109, ...",10


## 10. Top 3 funcionários por minutos

Os funcionários são ordenados de forma determinística:

1. `totalMinutes` em ordem decrescente;
2. `userId` em ordem crescente em caso de empate.

In [18]:
funcs_por_minutos = funcs_agrupadas.sort_values(
    by=["totalMinutes", "userId"],
    ascending=[False, True]
)

top_3_funcionarios = (
    funcs_por_minutos[["userId", "userName", "totalMinutes"]]
    .head(3)
    .to_dict(orient="records")
)

top_3_funcionarios = [
    {
        "userId": int(f["userId"]),
        "userName": f["userName"],
        "totalMinutes": int(f["totalMinutes"])
    }
    for f in top_3_funcionarios
]

df_top_3_funcionarios = pd.DataFrame(top_3_funcionarios)
df_top_3_funcionarios.head(3)

,userId,userName,totalMinutes
0,5,Eduardo,4303
1,1,Ana,4077
2,3,Carla,3842


## 11. Usuário com mais tarefas distintas

A ordenação segue a regra da atividade:

1. `distinctTasks` em ordem decrescente;
2. `userId` em ordem crescente em caso de empate.

In [19]:
funcs_por_tarefas = funcs_agrupadas.sort_values(
    by=["distinctTasks", "userId"],
    ascending=[False, True]
)

usuarios_mais_tarefas = funcs_por_tarefas.iloc[0].to_dict()

usuarios_mais_tarefas_limpo = {
    "userId": int(usuarios_mais_tarefas["userId"]),
    "userName": usuarios_mais_tarefas["userName"],
    "distinctTasks": int(usuarios_mais_tarefas["distinctTasks"]),
    "taskIds": [int(x) for x in usuarios_mais_tarefas["taskIds"]]
}

df_usuarios_mais_tarefas_limpo = pd.DataFrame([usuarios_mais_tarefas_limpo])
df_usuarios_mais_tarefas_limpo.head()

,userId,userName,distinctTasks,taskIds
0,1,Ana,10,"[101, 102, 103, 104, 105, 106, 107, 108, 109, ..."


---

# L — Carga

## 12. Montagem da estrutura final

Criação do dicionário final seguindo a estrutura exigida pela atividade.

In [21]:
resultado_final = {
    "totalMinutes": int(total_minutes),
    "tasks": tasks_ordenadas.to_dict(orient="records"),
    "mostWorkedTask": tarefa_mais_trabalhada_limpa,
    "top3TasksPercentage": top_3_tarefas,
    "top3Employees": top_3_funcionarios,
    "mostDistinctUserOnTasks": usuarios_mais_tarefas_limpo,
    "ignoredRecords": int(registros_ignorados)
}

resultado_final

{'totalMinutes': 28408,
 'tasks': [{'taskId': 103,
   'taskName': 'Ajustar layout',
   'totalMinutes': 4047,
   'percentage': '14.25%'},
  {'taskId': 110,
   'taskName': 'Criar endpoint relatório',
   'totalMinutes': 3500,
   'percentage': '12.32%'},
  {'taskId': 106,
   'taskName': 'Criar testes unitários',
   'totalMinutes': 3050,
   'percentage': '10.74%'},
  {'taskId': 108,
   'taskName': 'Melhorar queries',
   'totalMinutes': 3041,
   'percentage': '10.70%'},
  {'taskId': 105,
   'taskName': 'Refatorar autenticação',
   'totalMinutes': 2804,
   'percentage': '9.87%'},
  {'taskId': 104,
   'taskName': 'Documentar API',
   'totalMinutes': 2728,
   'percentage': '9.60%'},
  {'taskId': 102,
   'taskName': 'Criar tela dashboard',
   'totalMinutes': 2713,
   'percentage': '9.55%'},
  {'taskId': 107,
   'taskName': 'Implementar cache',
   'totalMinutes': 2474,
   'percentage': '8.71%'},
  {'taskId': 101,
   'taskName': 'Corrigir bug login',
   'totalMinutes': 2088,
   'percentage': '7.35

## 13. Geração do arquivo `result.json`

Salva o resultado final em JSON, preservando caracteres acentuados com `ensure_ascii=False`.

In [13]:
with open("result.json", "w", encoding="utf-8") as f:
    json.dump(resultado_final, f, indent=2, ensure_ascii=False)

print("Arquivo result.json gerado com sucesso!")

Arquivo result.json gerado com sucesso!


---

## 14. Conferência rápida

Resumo dos principais valores gerados para validação.

In [14]:
print(f"Total de minutos: {resultado_final['totalMinutes']}")
print(f"Registros ignorados: {resultado_final['ignoredRecords']}")
print(f"Tarefa mais trabalhada: {resultado_final['mostWorkedTask']}")
print(f"Top 3 funcionários: {resultado_final['top3Employees']}")

Total de minutos: 28408
Registros ignorados: 41
Tarefa mais trabalhada: {'taskId': 103, 'taskName': 'Ajustar layout', 'totalMinutes': 4047, 'percentage': '14.25%'}
Top 3 funcionários: [{'userId': 5, 'userName': 'Eduardo', 'totalMinutes': 4303}, {'userId': 1, 'userName': 'Ana', 'totalMinutes': 4077}, {'userId': 3, 'userName': 'Carla', 'totalMinutes': 3842}]
